# Modul 18: LeNet und Transfer Learning mit PyTorch

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** LeNet mit PyTorch, Transfer mit PyTorch  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittene PyTorch-Anwendung für Bilddaten  
    **Orientierungszeit:** etwa 160 bis 220 Minuten

    ## Überblick

    Sie bauen eine Bilddatenpipeline, implementieren ein LeNet-ähnliches nn.Module und analysieren Training sowie Fehlerbilder. Danach untersuchen Sie Augmentation, Dropout, BatchNorm, Scheduler und ein kleines Transfer-Learning-Experiment mit MobileNetV3 Small.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_18A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_18B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Bildtransformationen und DataLoader für Trainings-, Validierungs- und Testbilder definieren.
- Ausgabeformen von Conv2d, Pooling und Flatten im channels-first-Format bestimmen.
- Eine LeNet-Klasse mit nn.Module implementieren und auf CPU oder GPU trainieren.
- Fehlerbilder und Framework-spezifische Datenformen nachvollziehbar analysieren.
- Augmentation, Dropout, BatchNorm, Lernratenscheduler und Early Stopping kombinieren.
- MobileNetV3 Small laden, Feature-Schichten einfrieren und einen Klassifikationskopf anpassen.
- Genauigkeit, trainierbare Parameter, Inferenzzeit und Artefaktgröße gemeinsam bewerten.

    ## Bewertete Fähigkeiten

    - torchvision.transforms, Dataset und DataLoader
- Conv2d, MaxPool2d, Flatten und LeNet als nn.Module
- CNN-Training und Fehleranalyse
- Augmentation, BatchNorm, Dropout, Scheduler und Early Stopping
- MobileNetV3 Small, Freezing, Transfer-Kopf und Ressourcenvergleich

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames PyTorch-Setup
import io
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RANDOM_SEED = 42
FAST_MODE = os.environ.get("COURSE_FAST", "0") == "1"
OFFLINE_MODE = os.environ.get("COURSE_OFFLINE", "0") == "1"

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.use_deterministic_algorithms(False)
warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torchvision import models, transforms

digits_18 = load_digits()
images_18 = (digits_18.images.astype("float32") / 16.0)
labels_18 = digits_18.target.astype("int64")

X_train_valid_18, X_test_18, y_train_valid_18, y_test_18 = train_test_split(
    images_18,
    labels_18,
    test_size=0.20,
    stratify=labels_18,
    random_state=RANDOM_SEED,
)
X_train_18, X_valid_18, y_train_18, y_valid_18 = train_test_split(
    X_train_valid_18,
    y_train_valid_18,
    test_size=0.25,
    stratify=y_train_valid_18,
    random_state=RANDOM_SEED,
)

transfer_mask_18 = np.isin(labels_18, [0, 1, 2])
transfer_images_18 = images_18[transfer_mask_18]
transfer_labels_18 = labels_18[transfer_mask_18]
TX_train_valid_18, TX_test_18, Ty_train_valid_18, Ty_test_18 = train_test_split(
    transfer_images_18,
    transfer_labels_18,
    test_size=0.20,
    stratify=transfer_labels_18,
    random_state=RANDOM_SEED,
)
TX_train_18, TX_valid_18, Ty_train_18, Ty_valid_18 = train_test_split(
    TX_train_valid_18,
    Ty_train_valid_18,
    test_size=0.25,
    stratify=Ty_train_valid_18,
    random_state=RANDOM_SEED,
)

print("LeNet Train/Valid/Test:", X_train_18.shape, X_valid_18.shape, X_test_18.shape)
print("Transfer Train/Valid/Test:", TX_train_18.shape, TX_valid_18.shape, TX_test_18.shape)

print("PyTorch-Version:", torch.__version__)
print("Gerät:", DEVICE)


## Aufgabe 1: Bild-Dataset, Transformationen und Conv-Formen

    Erstellen Sie eine kleine PyTorch-Bilddatenpipeline.

1. Definieren Sie eine `Dataset`-Klasse, die NumPy-Bilder und Labels speichert.
2. Wandeln Sie jedes `8 x 8`-Bild in einen `float32`-Tensor der Form `(1, 8, 8)` um.
3. Erstellen Sie getrennte DataLoader. Mischen Sie nur das Training.
4. Prüfen Sie einen Batch auf die Form `(Batch, Kanäle, Höhe, Breite)`.
5. Wenden Sie testweise `Conv2d(1, 6, kernel_size=3, padding=1)` und `MaxPool2d(2)` an und bestätigen Sie die resultierenden Formen.
6. Erklären Sie den Formunterschied zum standardmäßigen Keras-Format.

> **Hinweis:** PyTorch erwartet die Kanalachse vor den räumlichen Achsen.

In [ ]:
batch_size_18 = 32

# Speichern Sie die Loader als train_loader_18, valid_loader_18 und
# test_loader_18 für die folgenden Aufgaben.

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** PyTorch erwartet die Kanalachse vor den räumlichen Achsen.

## Aufgabe 2: LeNet als nn.Module implementieren

    Implementieren Sie ein LeNet-ähnliches CNN für zehn Klassen.

1. Erstellen Sie zwei Faltungsblöcke mit ReLU und Max-Pooling.
2. Bestimmen Sie die Flatten-Größe durch eine kurze Formrechnung oder einen sicheren Hilfsdurchlauf.
3. Ergänzen Sie eine Dense-Schicht und eine lineare Ausgabeschicht mit zehn Logits.
4. Verschieben Sie das Modell auf `DEVICE`.
5. Richten Sie `CrossEntropyLoss` und Adam ein.
6. Prüfen Sie Ausgabeform und Parameterzahl.

> **Hinweis:** Verfolgen Sie die Form `1x8x8 -> 8x4x4 -> 16x2x2`.

In [ ]:
# Speichern Sie die Objekte als lenet_18, criterion_18 und optimizer_18.

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Verfolgen Sie die Form `1x8x8 -> 8x4x4 -> 16x2x2`.

## Aufgabe 3: CNN trainieren und Fehlerbilder untersuchen

    Trainieren und bewerten Sie `lenet_18`.

1. Schreiben Sie Trainings- und Evaluationsfunktionen für Mehrklassenklassifikation.
2. Trainieren Sie mit Early Stopping anhand des Validierungsverlusts.
3. Stellen Sie die besten Gewichte wieder her.
4. Visualisieren Sie Loss und Accuracy.
5. Berechnen Sie Testgenauigkeit und Konfusionsmatrix.
6. Zeigen Sie bis zu sechs falsch klassifizierte Bilder mit wahrem Label, Vorhersage und Konfidenz.

> **Hinweis:** Speichern Sie die besten Parameter als unabhängige CPU-Kopien.

In [ ]:
max_epochs_18 = 5 if FAST_MODE else 35
patience_18 = 6

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Speichern Sie die besten Parameter als unabhängige CPU-Kopien.

## Aufgabe 4: Augmentation, BatchNorm, Dropout und Scheduler kombinieren

    Erstellen Sie eine regulierte CNN-Variante.

1. Definieren Sie eine kleine `RandomAffine`-Transformation nur für das Training.
2. Erstellen Sie einen neuen Trainings-DataLoader mit dieser Transformation.
3. Ergänzen Sie BatchNorm nach den Faltungen und Dropout vor der Ausgabeschicht.
4. Verwenden Sie Adam und `ReduceLROnPlateau` auf dem Validierungsverlust.
5. Implementieren Sie weiterhin Early Stopping und speichern Sie die Lernrate pro Epoche.
6. Vergleichen Sie beste Validierungsgenauigkeit und Testgenauigkeit mit dem ersten LeNet-Modell.

> **Hinweis:** Rufen Sie den Scheduler nach der Validierungsphase mit dem überwachten Wert auf.

In [ ]:
regularized_epochs_18 = 5 if FAST_MODE else 35

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Rufen Sie den Scheduler nach der Validierungsphase mit dem überwachten Wert auf.

## Aufgabe 5: Integration: MobileNetV3 Small einfrieren und bewerten

    Führen Sie ein kleines Transfer-Learning-Projekt für die Klassen 0, 1 und 2 durch.

1. Erstellen Sie eine Transformation von `1 x 8 x 8` zu normalisierten `3 x 64 x 64`-Tensoren.
2. Laden Sie MobileNetV3 Small mit öffentlichen Standardgewichten, falls verfügbar. Verwenden Sie sonst einen dokumentierten Fallback.
3. Frieren Sie alle Feature-Schichten ein und ersetzen Sie die letzte Klassifikationsschicht durch drei Ausgaben.
4. Trainieren Sie nur den Klassifikationskopf für wenige Epochen.
5. Trainieren Sie ein kleines Scratch-CNN auf genau denselben transformierten Bildern und Splits.
6. Vergleichen Sie Validierungs- und Testgenauigkeit, trainierbare Parameter, Median-Inferenzzeit und Größe des `state_dict`.

> **Hinweis:** Zählen Sie nur Parameter mit `requires_grad=True`, wenn Sie den Trainingsaufwand vergleichen.

In [ ]:
transfer_epochs_18 = 1 if FAST_MODE else 3
transfer_batch_size_18 = 32

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Zählen Sie nur Parameter mit `requires_grad=True`, wenn Sie den Trainingsaufwand vergleichen.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.